In [ ]:
import os
import time
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv

True

In [ ]:
from alpaca.data.historical.news import NewsClient
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import NewsRequest, StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.data.enums import Adjustment, DataFeed

In [4]:
load_dotenv("../.env")

API_KEY = os.getenv("API_KEY")
SECRET_KEY = os.getenv("SECRET_KEY")

In [11]:
news_client = NewsClient(API_KEY, SECRET_KEY)
price_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

In [5]:
# TODO: Pick Stocks
tech = ['AAPL']
healthcare = ["JNJ"]
...
TICKERS = tech + healthcare

In [14]:
# YEARS = range(2020, 2027) #consider pandemic years
# END_DATE  = datetime(2026, 10, 16)  # refer to alpaca, or datetime.now().date()
START_DATE = datetime(2020, 1, 1)
END_DATE = datetime(2026, 10, 16)
BENCHMARK = "SPY"

In [7]:
# Output folders
NEWS_DIR = "../data/raw/news"
PRICES_PATH = "../data/raw/prices.parquet"

os.makedirs(NEWS_DIR, exist_ok=True)

In [8]:
# func requsting one ticker in alpaca sdk (refer to alpaca sdk doc)
def fetch_news(symbol, start, end):
    
    request = NewsRequest(symbols=symbol, start=start, end=end)
    
    response = news_client.get_news(request)

    rows = []
    for article in response.data["news"]:
        rows.append({
            "ticker": symbol,
            "created_at": article.created_at,
            "headline": article.headline,
            "source": article.source,
            "symbols": article.symbols,
        })
    return pd.DataFrame(rows)

In [12]:
# Check: return a df of all news between start and end
test = fetch_news("AAPL", datetime(2026, 9, 17), datetime(2026, 10, 17))
test.head()

,ticker,created_at,headline,source,symbols
0,AAPL,2026-10-16 15:33:56+00:00,'Apple Set For UK Legal Fight Over iPhone Tap-...,benzinga,"[AAPL, EWU]"
1,AAPL,2026-10-16 15:19:23+00:00,'PayPal And Epic Games Have Discussed Concerns...,benzinga,"[AAPL, PYPL]"
2,AAPL,2026-10-16 13:14:35+00:00,Apple Stock Hits Record High as iPhone Duo Pre...,benzinga,[AAPL]
3,AAPL,2026-10-16 13:11:24+00:00,October Rate-Hike Odds Collapse as Warsh Keeps...,benzinga,"[AAPL, SPY]"
4,AAPL,2026-10-16 12:38:48+00:00,"Apple’s $1,999 iPhone Duo Preorders Begin Toda...",benzinga,"[AAPL, T, TMUS, VZ]"


In [13]:
print(test["created_at"].min(), test["created_at"].max())

2026-09-17 07:14:14+00:00 2026-10-16 15:33:56+00:00


In [15]:
# pull all tickers
for ticker in TICKERS:
    out_path = f"{NEWS_DIR}/{ticker}.parquet"

    # just get rid of file already exists
    if os.path.exists(out_path):
        print(f"{ticker}: already saved, skipping")
        continue

    df_ticker = fetch_news(ticker, START_DATE, END_DATE)
    df_ticker.to_parquet(out_path)
    print(f"{ticker}: {len(df_ticker)} articles saved")

AAPL: 23674 articles saved
JNJ: 4219 articles saved


In [16]:
# Check stock price - refer to alpaca_exploration

# request = StockBarsRequest(
#     symbol_or_symbols=ticker,      
#     timeframe=TimeFrame.Hour,        
#     start=start,                     
#     end=end,
#     feed=DataFeed.IEX,
# )

price_request = StockBarsRequest(
    symbol_or_symbols=TICKERS + [BENCHMARK],
    timeframe=TimeFrame.Day,
    start=datetime(2020, 1, 1),
    end=END_DATE,
    adjustment=Adjustment.ALL,
    feed=DataFeed.SIP,
)


In [30]:
price_client.get_stock_bars(price_request).df.head()

open    high     low   close      volume  \
symbol timestamp                                                               
SPY    2020-01-02 05:00:00+00:00  294.17  295.39  293.25  295.38  60187033.0   
       2020-01-03 05:00:00+00:00  292.00  294.26  291.95  293.16  80319689.0   
       2020-01-06 05:00:00+00:00  291.39  294.34  291.28  294.34  56672077.0   
       2020-01-07 05:00:00+00:00  293.69  294.17  292.98  293.44  43646563.0   
       2020-01-08 05:00:00+00:00  293.62  296.20  293.38  294.97  69691471.0   

                                  trade_count    vwap  
symbol timestamp                                       
SPY    2020-01-02 05:00:00+00:00     304886.0  294.29  
       2020-01-03 05:00:00+00:00     358026.0  293.43  
       2020-01-06 05:00:00+00:00     255769.0  293.31  
       2020-01-07 05:00:00+00:00     226060.0  293.60  
       2020-01-08 05:00:00+00:00     340005.0  295.09

In [29]:
prices = price_client.get_stock_bars(price_request).df.reset_index()
prices.head()

,symbol,timestamp,open,high,low,close,volume,trade_count,vwap
0,AAPL,2020-01-02 05:00:00+00:00,71.28,72.33,71.03,72.27,146217352.0,283489.0,71.84
1,AAPL,2020-01-03 05:00:00+00:00,71.50,72.33,71.35,71.57,154711256.0,280441.0,71.91
2,AAPL,2020-01-06 05:00:00+00:00,70.69,72.18,70.44,72.14,125599968.0,242868.0,71.66
3,AAPL,2020-01-07 05:00:00+00:00,72.15,72.40,71.58,71.80,118227136.0,227871.0,71.96
4,AAPL,2020-01-08 05:00:00+00:00,71.50,73.26,71.50,72.95,142736804.0,261773.0,72.50


In [32]:
news_all = pd.concat(
    [pd.read_parquet(f"{NEWS_DIR}/{t}.parquet") for t in TICKERS],
    ignore_index=True,
)
print(len(news_all))

news_all.head()

27893


,ticker,created_at,headline,source,symbols
0,AAPL,2026-10-15 18:33:56+00:00,Anthropic Needs Nearly $2.7B in Additional Mon...,benzinga,"[AAPL, AMZN, BTCUSD, GOOG, META, MSFT, NVDA, Q..."
1,AAPL,2026-10-15 18:04:31+00:00,Michael Burry's 'Red October' Warning: Why QQQ...,benzinga,"[AAPL, AMZN, DIA, GOOGL, META, MSFT, NVDA, QQQ..."
2,AAPL,2026-10-15 17:26:44+00:00,"Meta Announces Release Of Muse Spark Open, The...",benzinga,"[AAPL, META, MSFT]"
3,AAPL,2026-10-15 17:04:44+00:00,'Apple and Meta team up to make Muse AI model ...,benzinga,"[AAPL, META, MSFT]"
4,AAPL,2026-10-15 16:47:46+00:00,Apple Hits $5 Trillion Market Cap as Nvidia's ...,benzinga,"[AAPL, CRWV, NVDA, ORCL, TSM]"


In [33]:
# articles per ticket
news_all.groupby("ticker").size().sort_values()

ticker
JNJ      4219
AAPL    23674
dtype: int64

In [35]:
# articles per year
news_all["year"] = pd.to_datetime(news_all["created_at"]).dt.year
news_all.groupby(["ticker", "year"]).size().unstack()

year,2020,2021,2022,2023,2024,2025,2026
ticker,,,,,,,
AAPL,2035,3645,4246,4272,3873,3318,2285
JNJ,578,1161,575,574,500,510,321


In [ ]:
# price coverage
prices.groupby("symbol")["timestamp"].agg(["min", "max", "count"]) #1706 trading days for now 

,min,max,count
symbol,,,
AAPL,2020-01-02 05:00:00+00:00,2026-10-15 04:00:00+00:00,1706
JNJ,2020-01-02 05:00:00+00:00,2026-10-15 04:00:00+00:00,1706
SPY,2020-01-02 05:00:00+00:00,2026-10-15 04:00:00+00:00,1706


## Additional collection — Joyce

Carmen's 18 original Alpaca cells above are preserved, including their saved outputs. Michael's earlier API exploration remains in the repository's `src/scratch/` folder.

I added the sections below to collect a separate sample for EDA:
- Yahoo daily prices: **AAPL, JNJ, SPY and AMZN**, from **January 1, 2020 (included) to October 16, 2026 (excluded)**, using New York session dates.
- EODHD public demo headlines: **AAPL and AMZN**, from **January 1, 2024 (included) to January 1, 2025 (excluded)**, using provider timestamps in UTC.

Run these cells separately from the Alpaca cells. They do not need Alpaca keys or any local helper modules. Collection is off by default. With it off, the last cell only shows a summary of files already on your computer; their presence does not validate them.

Yahoo's public chart interface is undocumented. EODHD's demo supports only selected symbols, and completing its requests does not prove that its historical archive is complete. Keep provider data local while the team checks academic reuse terms. These prices and headlines do not replace the original Alpaca files.

In [ ]:
from pathlib import Path
from datetime import date, datetime, time as midnight, timedelta, timezone
from zoneinfo import ZoneInfo
from urllib.parse import urlsplit
import hashlib
import json
import time
import pandas as pd
import requests

# Run from the project folder or src/. No credentials are read here.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
RUN_SUPPLEMENTAL_COLLECTION = False
PRICE_SYMBOLS = ("AAPL", "JNJ", "SPY", "AMZN")
NEWS_SYMBOLS = ("AAPL", "AMZN")
PRICE_START, PRICE_END = "2020-01-01", "2026-10-16"
NEWS_START, NEWS_END = "2024-01-01", "2025-01-01"
YAHOO_DIR = PROJECT_ROOT / "data/supplemental/yahoo_prices"
NEWS_DIR = PROJECT_ROOT / "data/supplemental/eodhd_news"

# Fresh requests when enabled; existing local tables are not treated as a cache.
# A failed/incomplete run keeps previous normalized tables unchanged.

### Request records and retries

Each response is saved with its request parameters, retrieval time and SHA-256 hash. Requests are at least 1.1 seconds apart. Temporary connection errors and selected server errors get at most three attempts. A 403 or 429 stops collection immediately. Raw responses and metadata stay under `data/supplemental/`.

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()


def save_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")


_last_request_finished = None


def get_json(url, params, directory, tag, manifest):
    global _last_request_finished
    raw_dir = directory / "raw" / manifest["run_id"]
    raw_dir.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, 4):
        if _last_request_finished is not None:
            time.sleep(max(0, 1.1 - (time.monotonic() - _last_request_finished)))
        try:
            response = requests.get(url, params=params, timeout=(15, 60),
                                    allow_redirects=False,
                                    headers={"User-Agent": "FinancialNewsCourseProject/1.0"})
        except (requests.ConnectionError, requests.Timeout):
            _last_request_finished = time.monotonic()
            if attempt == 3:
                raise
            time.sleep(2 ** (attempt - 1))
            continue
        _last_request_finished = time.monotonic()
        raw_path = raw_dir / f"{tag}_attempt{attempt}.response"
        raw_path.write_bytes(response.content)
        metadata = {
            "endpoint": url, "parameters": params, "retrieved_at_utc": utc_now(),
            "http_status": response.status_code,
            "sha256": hashlib.sha256(response.content).hexdigest(),
            "raw_file": str(raw_path.relative_to(PROJECT_ROOT)),
        }
        save_json(raw_path.with_suffix(".request.json"), metadata)
        manifest["responses"].append(metadata)
        if response.status_code in (403, 429):
            raise RuntimeError(f"HTTP {response.status_code}: collection stopped; do not retry the gate.")
        if response.status_code in (500, 502, 503, 504) and attempt < 3:
            time.sleep(2 ** (attempt - 1))
            continue
        if response.status_code != 200:
            raise RuntimeError(f"HTTP {response.status_code}: request failed.")
        return response.json(), metadata


def new_manifest(provider):
    return {"provider": provider, "run_id": datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"),
            "started_at_utc": utc_now(), "status": "collecting", "responses": []}


def save_table(frame, directory, filename, manifest):
    directory.mkdir(parents=True, exist_ok=True)
    output = directory / filename
    temporary = output.with_suffix(".tmp.parquet")
    try:
        frame.to_parquet(temporary, index=False)
        temporary.replace(output)
    finally:
        temporary.unlink(missing_ok=True)
    manifest.update({"rows": len(frame), "file": str(output.relative_to(PROJECT_ROOT)),
                     "sha256": hashlib.sha256(output.read_bytes()).hexdigest()})

### Yahoo daily prices

The request uses New York midnight boundaries and includes adjusted close. The supplied OHLC fields remain separate from adjusted close; no additional price adjustment or missing-value filling is applied.

In [ ]:
def collect_yahoo_prices():
    directory = YAHOO_DIR
    manifest = new_manifest("Yahoo public chart")
    ny = ZoneInfo("America/New_York")
    params = {
        "period1": int(datetime.combine(date.fromisoformat(PRICE_START), midnight.min, ny).timestamp()),
        "period2": int(datetime.combine(date.fromisoformat(PRICE_END), midnight.min, ny).timestamp()),
        "interval": "1d", "events": "div,splits", "includeAdjustedClose": "true",
    }
    manifest.update({"symbols": list(PRICE_SYMBOLS), "start_inclusive_ny": PRICE_START,
                     "end_exclusive_ny": PRICE_END})
    frames = []
    try:
        for symbol in PRICE_SYMBOLS:
            payload, meta = get_json(
                f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}",
                params, directory, symbol, manifest)
            chart = payload.get("chart", {})
            if chart.get("error") or not chart.get("result"):
                raise ValueError(f"{symbol}: Yahoo did not return a price result.")
            result = chart["result"][0]
            timestamps = result.get("timestamp", [])
            quote = result.get("indicators", {}).get("quote", [{}])[0]
            adjusted = result.get("indicators", {}).get("adjclose", [{}])[0].get("adjclose", [])
            arrays = {name: quote.get(name, []) for name in ("open", "high", "low", "close", "volume")}
            arrays["adjusted_close"] = adjusted
            if not timestamps or any(len(values) != len(timestamps) for values in arrays.values()):
                raise ValueError(f"{symbol}: missing or unequal-length price fields.")
            frame = pd.DataFrame(arrays)
            frame["timestamp"] = pd.to_datetime(timestamps, unit="s", utc=True)
            frame["date"] = frame.timestamp.dt.tz_convert(ny).dt.tz_localize(None).dt.normalize()
            frame["symbol"] = symbol
            frame["retrieved_at"] = meta["retrieved_at_utc"]
            frame["raw_response_sha256"] = meta["sha256"]
            frame = frame.loc[frame.date.ge(pd.Timestamp(PRICE_START)) & frame.date.lt(pd.Timestamp(PRICE_END))]
            if frame.empty or frame.date.duplicated().any() or frame[list(arrays)].isna().any().any():
                raise ValueError(f"{symbol}: empty, duplicate-date or missing-value rows; inspect raw response.")
            if frame["adjusted_close"].le(0).any():
                raise ValueError(f"{symbol}: non-positive adjusted close.")
            frames.append(frame)
        combined = pd.concat(frames, ignore_index=True).sort_values(["symbol", "date"])
        save_table(combined, directory, "prices.parquet", manifest)
        manifest["status"] = "requests_completed_price_rows_saved"
    except Exception as error:
        manifest.update({"status": "failed_previous_price_table_unchanged", "error": str(error)})
        raise
    finally:
        manifest["finished_at_utc"] = utc_now()
        save_json(directory / "collection_run.json", manifest)
    return combined

### EODHD headlines

Requests are split by ticker and calendar month. Each page asks for up to 1,000 records, with offsets from 0 through 9,000. A repeated page or exhausted page limit is treated as incomplete collection. Publication timestamps are kept as supplied and converted to UTC; invalid or out-of-window dates stop the run. This still does not establish when revised headlines or sentiment scores were historically available.

In [ ]:
def month_windows(start, end):
    current, stop = date.fromisoformat(start), date.fromisoformat(end)
    while current < stop:
        following = date(current.year + (current.month == 12), current.month % 12 + 1, 1)
        upper = min(following, stop)
        yield current, upper
        current = upper


def collect_eodhd_news():
    directory = NEWS_DIR
    manifest = new_manifest("EODHD public demo")
    manifest.update({"symbols": list(NEWS_SYMBOLS), "start_inclusive_utc": NEWS_START,
                     "end_exclusive_utc": NEWS_END, "windows": []})
    rows = []
    try:
        for ticker in NEWS_SYMBOLS:
            for start, end in month_windows(NEWS_START, NEWS_END):
                lower, upper = pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC")
                window = {"ticker": ticker, "start": str(start), "end_exclusive": str(end), "response_rows": 0}
                manifest["windows"].append(window)
                seen_pages = set()
                for offset in range(0, 10000, 1000):
                    params = {"api_token": "demo", "s": ticker + ".US", "from": str(start),
                              "to": str(end - timedelta(days=1)), "limit": 1000, "offset": offset, "fmt": "json"}
                    payload, meta = get_json("https://eodhd.com/api/news", params, directory,
                                             f"{ticker}_{start}_{offset}", manifest)
                    if not isinstance(payload, list) or len(payload) > 1000:
                        raise ValueError("Unexpected EODHD response schema or page size.")
                    fingerprint = hashlib.sha256(json.dumps(sorted(json.dumps(x, sort_keys=True) for x in payload)).encode()).hexdigest()
                    if payload and fingerprint in seen_pages:
                        raise ValueError("Repeated EODHD page; pagination could not be confirmed.")
                    seen_pages.add(fingerprint)
                    window["response_rows"] += len(payload)
                    for article in payload:
                        timestamp = pd.Timestamp(article.get("date"))
                        if pd.isna(timestamp) or timestamp.tzinfo is None:
                            raise ValueError("Missing or timezone-naive news timestamp; inspect raw response.")
                        timestamp = timestamp.tz_convert("UTC")
                        if not lower <= timestamp < upper:
                            raise ValueError("News timestamp outside its requested month.")
                        headline, link = article.get("title"), article.get("link")
                        parsed = urlsplit(link or "")
                        sentiment = article.get("sentiment") if isinstance(article.get("sentiment"), dict) else {}
                        rows.append({
                            "ticker": ticker, "created_at": timestamp, "headline": headline, "link": link,
                            "source_domain": parsed.hostname if parsed.scheme in ("http", "https") else None,
                            "symbols": article.get("symbols") if isinstance(article.get("symbols"), list) else [],
                            "tags": article.get("tags") if isinstance(article.get("tags"), list) else [],
                            **{f"sentiment_{key}": sentiment.get(key) for key in ("polarity", "neg", "neu", "pos")},
                            "provider": "EODHD", "retrieved_at": meta["retrieved_at_utc"],
                            "query_window_start": str(start), "query_window_end": str(end),
                            "raw_response_sha256": meta["sha256"],
                        })
                    if len(payload) < 1000:
                        window["status"] = "requests_completed_archive_unverified"
                        break
                else:
                    raise ValueError("EODHD offset limit reached; month may be incomplete.")
        columns = ["ticker", "created_at", "headline", "link", "source_domain", "symbols", "tags",
                   "sentiment_polarity", "sentiment_neg", "sentiment_neu", "sentiment_pos",
                   "provider", "retrieved_at", "query_window_start", "query_window_end", "raw_response_sha256"]
        frame = pd.DataFrame(rows, columns=columns)
        frame["created_at"] = pd.to_datetime(frame["created_at"], utc=True)
        frame["retrieved_at"] = pd.to_datetime(frame["retrieved_at"], utc=True)
        # Deduplicate only complete keys. Do not merge the same URL across different stocks.
        key = ["ticker", "link", "headline", "created_at"]
        duplicates = frame[key].notna().all(axis=1) & frame.duplicated(key)
        manifest["exact_duplicate_copies_removed"] = int(duplicates.sum())
        frame = frame.loc[~duplicates].sort_values(["ticker", "created_at"]).reset_index(drop=True)
        save_table(frame, directory, "news.parquet", manifest)
        manifest["status"] = "requests_completed_archive_unverified"
        manifest["archive_note"] = "Pagination completed for requested windows; historical completeness is unverified."
    except Exception as error:
        manifest.update({"status": "failed_or_incomplete_previous_news_table_unchanged", "error": str(error)})
        raise
    finally:
        manifest["finished_at_utc"] = utc_now()
        save_json(directory / "collection_run.json", manifest)
    return frame

In [ ]:
if RUN_SUPPLEMENTAL_COLLECTION:
    collected_prices = collect_yahoo_prices()
    collected_news = collect_eodhd_news()
    print(f"Saved {len(collected_prices):,} price rows and {len(collected_news):,} ticker-news rows.")
else:
    print("Additional collection is off. No HTTP requests were made.")

# Local summaries only. Existing files are not validated or refreshed by this step.
for label, path, symbol_column, time_column in [
    ("Yahoo prices", YAHOO_DIR / "prices.parquet", "symbol", "date"),
    ("EODHD news", NEWS_DIR / "news.parquet", "ticker", "created_at"),
]:
    print(label + ":", path.relative_to(PROJECT_ROOT))
    if path.is_file():
        local = pd.read_parquet(path)
        print(local.groupby(symbol_column)[time_column].agg(["count", "min", "max"]))
    else:
        print("No local file yet.")

The EDA uses the saved tables at `data/supplemental/yahoo_prices/prices.parquet` and `data/supplemental/eodhd_news/news.parquet`. All response files and metadata stay in the same provider folders. No additional collection code runs merely by defining the functions above. To collect, change `RUN_SUPPLEMENTAL_COLLECTION` to `True` and run only this added section.